# CLB
## Dependencies

In [1]:
import os
import tqdm
import shutil
import numpy as np
import scipy.io
import pandas as pd
import pickle
import h5py

import logging

In [2]:
def load_mocap_data(file_path):
    mat = scipy.io.loadmat(file_path)
    exp_name = [m for m in mat.keys() if m[:2] != '__'][0]  ## TOCHANGE
    data = np.moveaxis(mat[exp_name][0, 0]['Trajectories'][0, 0]['Labeled']['Data'][0, 0], 2, 0)
    keypoints = [label[0].replace('coordinate', 'coord') for label in
                    mat[exp_name][0, 0]['Trajectories'][0, 0]['Labeled']['Labels'][0, 0][0]]

    # Very important. Make sure the keypoints are always in the same order even if not saved so in the original files
    new_order = np.argsort(keypoints)
    keypoints = [keypoints[n] for n in new_order]
    data = data[:, new_order, :]

    return data, keypoints

## Load data

In [3]:
# check file names
file_path = "/home/group-cvg/cvg-rose/bogna_mocap/CP1A_CLB/"
files = os.listdir(file_path)
files

['CP1_S13_M14_MC2_CLB_21_09_2018_1300_proc_bij_19.11.2018_A.mat',
 'CP1_S7_M1_MC3_CLB_14_09_2018_1430__proc_bij_15.11.2018_A.mat',
 'CP1_S12_M19_MC3_CLB_18_09_2018_1600_proc_bij_19.11.2018_A.mat',
 'CP1_S16_M19_MC3_CLB_21_09_2018_1530_bij_proc_23.11.2018_D.mat',
 'CP1_S14_M15_MC2_CLB_21_09_2018_1400_proc_bij_19.11.2018_B.mat',
 'CP1_S15_M1_MC3_CLB_21_09_2018_1500_proc_bij_20.11.2018_C.mat',
 'CP1_S6_M15_MC2_CLB_14_09_2018_1100_proc_bij_13.11.2018_D.mat',
 'CP1_S5_M14_MC2_CLB_14_09_2018_1000_proc_bij_26.11.2018_C.mat',
 'CP1_S9_M14_MC2_CLB_18_09_2018_1330_proc_bij_16.11.2018_B.mat',
 'CP1_S10_M15_MC2_CLB_18_09_2018_1430_proc_bij_16.11.2018_C.mat',
 'CP1_S2_M15_MC2_CLB_12_09_2018_1350_proc_bij_9.11.2018_A.mat',
 'CP1_S3_M1_MC3_CLB_12_09_2018_1450_proc_bij_12.11.2018_B.mat',
 'CP1_S8_M19_MC2_CLB_14_09_2018_1530_proc_bij_15.11.2018_B.mat',
 'CP1_S1_M14_MC2_CLB_12_09_2018_1300_proc_bij_8.11.2018_D.mat',
 'CP1_S4_M19_MC2_CLB_12_09_2018_1540_proc_bij_13.11.2018_C.mat',
 'CP1_S11_M1_MC3_CLB_18

## Process

In [11]:
data_CP1A_CLB = {}

for file_name in files:
    file = os.path.join(file_path, file_name)
    
    if os.path.isfile(file):
        data, keypoints = load_mocap_data(file)
        key_name = str(file_name.split("_")[0] + "_" + file_name.split("_")[1]) + "_CLB"
        data_CP1A_CLB[key_name] ={"label":None, "keypoints":None}
        data_CP1A_CLB[key_name]["label"] = file_name.split(".")[-2][-1]
        data_CP1A_CLB[key_name]["keypoints"] = keypoints
        # 1. Discard last dimension
        data = data[ : , : , :3]
        
        # 2. Only take 36000 frames from each sequence
        l = len(data)
        r = l - 34500
        if r < 1500:
            data = data[r:l]
        else: # length  >= 37500
            data = data[1500: 36000]
        
        data_CP1A_CLB[key_name]["data"] = data

In [13]:
# The correct keypoint 
keypoints_name = ['left_ankle',  'left_back',  'left_coord',  'left_hip',  'left_knee', 
                  'right_ankle', 'right_back', 'right_coord', 'right_hip', 'right_knee']

for key, value in data_CP1A_CLB.items():
    # Delete arm keypoints
    arm_indices = [i for i, s in enumerate(value["keypoints"]) if "arm" in s]
    if len(arm_indices)> 0:
        for index in sorted(arm_indices, reverse=True):
            value["keypoints"].pop(index) 
            value["data"] = np.delete(value["data"], index, axis=1)
    
    # Create an empty array to store
    arr = np.full((34500, 10, 3), np.nan)
    i = 0  # index of correct keypoint
    j = 0  # index of actual keypoint
    while i < 10:
        if keypoints_name[i] == value["keypoints"][i]:
            arr[:, i, :] = value["data"][:, j, :]
            i += 1 
            j += 1
        else:
            value["keypoints"].insert(i, None)
            i += 1
    ################################################## 
    # Sample 5 sequences from the original sequences 
    #value["data"] = arr.reshape(-1, 5, 10, 3).transpose(1, 0, 2, 3) 
    ##################################################
    value["data"] = arr

In [19]:
for key, value in data_CP1A_CLB.items():
    print(key, value["label"], value["keypoints"], value["data"].shape)

CP1_S13_CLB A ['left_ankle', 'left_back', None, 'left_hip', 'left_knee', 'right_ankle', 'right_back', None, 'right_hip', 'right_knee'] (34500, 10, 3)
CP1_S7_CLB A ['left_ankle', 'left_back', None, 'left_hip', 'left_knee', 'right_ankle', 'right_back', None, 'right_hip', 'right_knee'] (34500, 10, 3)
CP1_S12_CLB A ['left_ankle', None, None, 'left_hip', 'left_knee', 'right_ankle', None, None, 'right_hip', 'right_knee'] (34500, 10, 3)
CP1_S16_CLB D ['left_ankle', None, None, 'left_hip', 'left_knee', 'right_ankle', None, None, 'right_hip', 'right_knee'] (34500, 10, 3)
CP1_S14_CLB B ['left_ankle', 'left_back', None, 'left_hip', 'left_knee', 'right_ankle', 'right_back', None, 'right_hip', 'right_knee'] (34500, 10, 3)
CP1_S15_CLB C ['left_ankle', None, None, 'left_hip', 'left_knee', 'right_ankle', None, None, 'right_hip', 'right_knee'] (34500, 10, 3)
CP1_S6_CLB D ['left_ankle', 'left_back', None, 'left_hip', 'left_knee', 'right_ankle', 'right_back', None, 'right_hip', 'right_knee'] (34500, 10, 

In [16]:
import numpy as np
test = data_CP1A_CLB["CP1_S5_CLB"]["data"][0]
print(np.nanmin(test[:,  0]))
print(np.nanmax(test[:,  0]))

73.70038421935992
90.88218067799897


In [13]:
print(np.nanmin(test[:, :, 1]))
print(np.nanmax(test[:, :, 1]))

-74.83260970767054
33.867599382519714


In [14]:
print(np.nanmin(test[:, :, 2]))
print(np.nanmax(test[:, :, 2]))

-58.70849040172543
167.04657969327022


## Export

In [18]:
import pickle

with open('../../data/data_CP1A_CLB.pkl', 'wb') as file:
    pickle.dump(data_CP1A_CLB, file)

In [2]:
import pickle


with open('/home/guo/myfolder/mocap/data/data_CP1A_CLB.pkl', 'rb') as file:
    data_CP1A_CLB = pickle.load(file)

## Draft

In [ ]:
filename = "CP1_S14_M15_MC2_CLB_21_09_2018_1400_proc_bij_19.11.2018_B.mat"
file_path = os.path.join("/home/group-cvg/cvg-rose/bogna_mocap/CP1A_CLB/", filename)
data, keypoint = load_mocap_data(file_path)
data = data[1500:36000,:,:3]

(34500, 10, 3)


In [23]:
print(len(data))
print(data.shape)
missing_per_sample = np.isnan(data).sum(axis=(1, 2))

34500
(34500, 10, 3)


In [28]:
print(np.nanmin(data[:, :, 0]))
print(np.nanmax(data[:, :, 0]))

-96.56138874253152
98.05687059107537


In [29]:
print(np.nanmin(data[:, :, 1]))
print(np.nanmax(data[:, :, 1]))

-49.613540096851104
191.9221832046774


In [30]:
print(np.nanmin(data[:, :, 2]))
print(np.nanmax(data[:, :, 2]))

-43.595406579600656
137.96418615162688
